[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_strain.ipynb)

# Two-Phase Composite RVE — Linear-Elastic Strain Solve

A minimal walkthrough of FFTjax's strain-based Newton-CG elastic solver
(`solvers.mechanical.strain_nw_cg.solve_elastic`) on a **two-phase composite**: a glass-fibre
reinforcement in an epoxy matrix, arranged in a square-packed pattern via
`generation.rve.make_square_composite_rve`, under a prescribed macroscopic strain.

Because the two phases have a large stiffness contrast (~23x), the reference-medium correction is
nontrivial — the Newton-CG solve actually iterates, redistributing stress between the stiff fibres
and the compliant matrix.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    print("Running locally — using the local src/ checkout.")

In [ ]:
import sys
sys.path.insert(0, "../src")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np

from generation.rve import make_square_composite_rve
from operators.green import build_freq_grid, build_green_operator
from mat_models.elastic import LinearElasticIsotropic, assemble_C_field
from solvers.mechanical.strain_nw_cg import solve_elastic

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

`generation.rve.make_square_composite_rve` builds a square-packed 2-fibre RVE: a matrix phase with
circular fibre cross-sections arranged on a square lattice, extruded along Z into a 3-D voxel grid.

In [ ]:
phase_np, N, n, L, phi_act = make_square_composite_rve(
    phi=0.5, r_fiber_um=5.0, spacing_um=0.5, N_min=32, nz=32,
)
Nv = int(np.prod(n))

print("grid n :", n)
print("domain L [um]:", tuple(float(Li) for Li in L))
print("fibre volume fraction (actual):", phi_act)

### Materials and stiffness field

A glass fibre in an epoxy matrix — a common, high-contrast (~23x stiffness ratio) composite.

In [ ]:
matrix = LinearElasticIsotropic(E=3.0e3,  nu=0.35, name="epoxy matrix")
fiber  = LinearElasticIsotropic(E=70.0e3, nu=0.20, name="glass fibre")

phase = jnp.array(phase_np.reshape(-1))   # 0 = matrix, 1 = fibre
C_field = assemble_C_field([matrix, fiber], phase)

print(matrix)
print(fiber)

### Frequency grid, Green's operator, and solve

The reference medium is the average of the two phases' Lamé parameters — a reasonable choice when
neither phase dominates.

In [ ]:
L_um = tuple(float(Li) for Li in L)
xi_flat = build_freq_grid(n, L_um)

lam0 = 0.5 * (matrix.lam + fiber.lam)
mu0  = 0.5 * (matrix.mu  + fiber.mu)
G_glob = build_green_operator(xi_flat, lam0, mu0)

eps_bar = jnp.array([
    [1.0e-3, 0.0, 0.0],
    [0.0,    0.0, 0.0],
    [0.0,    0.0, 0.0],
])

eps, sigma, delta, it, converged = solve_elastic(
    n, C_field, G_glob, eps_bar, toler_lin=1e-6, maxiter=1000,
)

print("CG iterations:", int(it))
print("converged     :", bool(converged))
print("sigma11 (avg) :", float(jnp.mean(sigma[0, 0])), "MPa")

The Newton-CG solve takes real iterations to converge — the correction field is doing real work redistributing stress between the stiff fibres and the compliant matrix.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(phase_np[:, :, 0].T, origin="lower", cmap="gray_r")
ax.set_title(f"Fibre cross-section (Vf={phi_act:.3f})")
ax.set_xlabel("x [voxel]")
ax.set_ylabel("y [voxel]")
plt.show()

## Next steps

- Sweep grid size for the composite RVE above — the [Benchmark](https://choROPeNt.github.io/FFTjax/documentation/benchmark#linear-elastic-strain-solve) page does exactly this and times it.
- See `solvers.mechanical.strain_nw_cg.dstrain_nw_cg_mixed` for mixed strain/stress   macroscopic boundary conditions.